# Notebook 06 — Observability

Shows how per-request traces make workflow behavior inspectable.

<!-- TODO main-session: expand intro -->

---

## Setup

Loads the repo root, environment, and public observability APIs used throughout this notebook.

<!-- TODO main-session: expand teaching framing -->

---

In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.workflow import run_workflow
from src.llm import LLMClient
from src.rag import ingest
from src.observability import (
    Span,
    Trace,
    SessionMetrics,
    LocalTraceStore,
    get_store,
    trace_workflow,
    compute_metrics,
)

has_key = bool(os.getenv("ANTHROPIC_API_KEY"))
print(f"Anthropic key present: {has_key}")

Anthropic key present: False


## Corpus

Indexes the sample program corpus into a notebook-local vector store for the traced workflow call.

<!-- TODO main-session: expand teaching framing -->

---

In [2]:
persist_dir = repo_root / "data" / "chroma_nb06"
result = ingest(repo_root / "data", persist_dir)
print(result)

'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 0ade70da-b9f4-43fd-b829-0073d0a2db2c)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json


Retrying in 1s [Retry 1/5].


'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: f57d8eb8-adba-4ec2-861d-0d5726298f9e)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json


Retrying in 2s [Retry 2/5].


'(ProtocolError('Connection aborted.', ConnectionResetError(10054, 'An existing connection was forcibly closed by the remote host', None, 10054, None)), '(Request ID: 52fbc82b-db6b-4335-b030-5f19df2f1a16)')' thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json


...

Retrying in 2s [Retry 2/5].


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


IngestionResult(documents_loaded=5, chunks_created=42, chunks_indexed=42, vector_store_path=WindowsPath('C:/Users/narla/OneDrive/Desktop/TalentSprint/IISc_GenAI_C2/LLMOps/llmops-session/data/chroma_nb06'), embedding_model='sentence-transformers/all-MiniLM-L6-v2')


## One traced workflow call

Wraps a single workflow run so the trace and its top-level observability fields are visible together.

<!-- TODO main-session: expand teaching framing -->

---

In [3]:
from IPython.display import display
import pandas as pd

llm = LLMClient() if has_key else LLMClient(provider="mock")
store = LocalTraceStore()
trace = trace_workflow(
    run_workflow,
    question="What is the late submission policy?",
    persist_dir=persist_dir,
    llm=llm,
    store=store,
)

if not has_key:
    print("Using mock client; token counts may be zero.")

display(trace)
with pd.option_context("display.max_columns", None, "display.max_colwidth", None):
    display(store.to_dataframe())

C:\Users\narla\AppData\Local\Programs\Python\Python312\Lib\site-packages\langgraph\checkpoint\base\__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Unexpected classification returned by LLM; falling back to out_of_scope. raw_output='[mock:59bb19f5] echo: You are a question classifier for the TalentSpri...'


Workflow trace entries did not expose cache_status; leaving Trace.cache_status empty.


Using mock client; token counts may be zero.


Trace(trace_id='254bd01d', query='What is the late submission policy?', start_ms=321158957.9985, spans=[Span(name='classify', start_ms=321158957.9985, end_ms=321158958.1146, attributes={'input': 'What is the late submission policy?', 'output': 'out_of_scope', 'latency_ms': 0.11610001092776656, 'prompt_version': 'v1', 'route_to': 'refuse'}), Span(name='refuse', start_ms=321158958.1146, end_ms=321158958.88, attributes={'input': 'What is the late submission policy?', 'output': 'Thanks for asking, but that question is outside the scope of this program assistant.\nPlease use the appropriate professional or official resource for:\n"What is the late submission policy?"\n', 'classification': 'out_of_scope', 'latency_ms': 0.7654000073671341, 'prompt_version': 'v1'})], category='out_of_scope', backend='mock', model='mock-model-v1', prompt_tokens=0, completion_tokens=0, total_tokens=0, latency_ms=282.2, retrieved_count=0, cache_status='', refused=True, escalated=False, guardrail_input_flag='', gu

,trace_id,query,category,backend,model,prompt_tokens,completion_tokens,total_tokens,latency_ms,retrieved_count,cache_status,refused,escalated,guardrail_input_flag,guardrail_output_flag,workflow_steps
0,254bd01d,What is the late submission policy?,out_of_scope,mock,mock-model-v1,0,0,0,282.2,0,,True,False,,,classify → refuse


## Spans — the per-node narrative inside the trace

Prints the ordered spans captured for the traced workflow call above.

<!-- TODO main-session: expand -->

In [4]:
for i, span in enumerate(trace.spans):
    print(
        f"[{i}] {span.name:15s} duration_ms={span.duration_ms:7.2f}  "
        f"attributes={dict(span.attributes)}"
    )
print(f"\nworkflow_steps = {trace.workflow_steps}")

[0] classify        duration_ms=   0.12  attributes={'input': 'What is the late submission policy?', 'output': 'out_of_scope', 'latency_ms': 0.11610001092776656, 'prompt_version': 'v1', 'route_to': 'refuse'}
[1] refuse          duration_ms=   0.77  attributes={'input': 'What is the late submission policy?', 'output': 'Thanks for asking, but that question is outside the scope of this program assistant.\nPlease use the appropriate professional or official resource for:\n"What is the late submission policy?"\n', 'classification': 'out_of_scope', 'latency_ms': 0.7654000073671341, 'prompt_version': 'v1'}

workflow_steps = ['classify', 'refuse']


## What each span attribute means

PLAN.md S-12 defines the per-node attributes that make the workflow debuggable from a saved trace.

<!-- TODO main-session: expand -->

## Cache hits — visible in cache_status

Runs the same question twice so the second trace can expose cache reuse directly.

<!-- TODO main-session: expand -->

In [5]:
import json
import time


class NotebookCacheDemoProvider:
    def __init__(self, model: str = "nb06-cache-demo-model") -> None:
        self.model = model

    def complete(
        self,
        prompt: str,
        system: str | None = None,
        **kwargs: object,
    ) -> dict[str, object]:
        time.sleep(0.05)
        if system and "Reply with exactly one category label." in system:
            text = "policy_question"
        elif "Respond as JSON with this schema:" in prompt:
            text = json.dumps(
                {
                    "answer": "Late submissions are allowed for two grace days.",
                    "grounded": True,
                    "source_section": "Late Submission Policy",
                    "confidence": 0.93,
                }
            )
        else:
            text = "out_of_scope"

        return {
            "text": text,
            "tokens_in": max(1, len((system or "") + prompt) // 4),
            "tokens_out": max(1, len(text) // 4),
            "raw": {"system": system, "kwargs": kwargs},
            "model": self.model,
        }


class NotebookCachedWorkflowLLM(LLMClient):
    def __init__(self) -> None:
        super().__init__(provider="mock", cache=True, prompt_version="nb06_cache_demo_v1")
        self._provider = NotebookCacheDemoProvider(model=self.model_name)
        self._provider_name = "notebook-cache-demo"


cache_demo_llm = NotebookCachedWorkflowLLM()
cache_store = LocalTraceStore()
question = "What is the late submission policy?"

trace1 = trace_workflow(
    run_workflow,
    question=question,
    persist_dir=persist_dir,
    llm=cache_demo_llm,
    store=cache_store,
)
trace2 = trace_workflow(
    run_workflow,
    question=question,
    persist_dir=persist_dir,
    llm=cache_demo_llm,
    store=cache_store,
)

for label, t in [("first call (miss)", trace1), ("second call (hit?)", trace2)]:
    print(f"\n=== {label} ===")
    print(f"  trace.cache_status = {t.cache_status!r}")
    print(f"  trace.total_tokens = {t.total_tokens}")
    print(f"  trace.latency_ms   = {t.latency_ms}")
    for span in t.spans:
        cs = span.attributes.get("cache_status", "—")
        tk = span.attributes.get("total_tokens", "—")
        print(f"    [{span.name}] cache_status={cs}  total_tokens={tk}")

print(f"\ntrace2 faster than trace1 = {trace2.latency_ms < trace1.latency_ms}")
if not has_key:
    print(
        "Notebook note: cell 6 stayed on the plain mock path, so this cache demo uses "
        "a notebook-local cached workflow client to keep the workflow offline while still "
        "producing a real answered trace."
    )


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



=== first call (miss) ===
  trace.cache_status = 'miss'
  trace.total_tokens = 0
  trace.latency_ms   = 733.0
    [classify] cache_status=—  total_tokens=—
    [retrieve] cache_status=—  total_tokens=—
    [answer] cache_status=miss  total_tokens=—

=== second call (hit?) ===
  trace.cache_status = 'hit'
  trace.total_tokens = 0
  trace.latency_ms   = 211.5
    [classify] cache_status=—  total_tokens=—
    [retrieve] cache_status=—  total_tokens=—
    [answer] cache_status=hit  total_tokens=—

trace2 faster than trace1 = True
Notebook note: cell 6 stayed on the plain mock path, so this cache demo uses a notebook-local cached workflow client to keep the workflow offline while still producing a real answered trace.


## The 11th field pays off

D-012 makes cache visibility a first-class trace field instead of something you have to infer indirectly.

<!-- TODO main-session: expand -->